In [3]:
import pandas as pd
import numpy as np

# 1. 파일 로드
file_path = "AUS_v2_Golden_Training_Set.csv"
print(f"🔍 [{file_path}] 정밀 진단 시작...")
print("-" * 80)

try:
    df = pd.read_csv(file_path, dtype={'COMPANY_ID_NORM': str})
    
    # [Check 1] 전체 컬럼 명단 및 타입 확인
    print("1️⃣ [Column & Type Check]")
    info_df = pd.DataFrame({
        'Type': df.dtypes,
        'Non-Null Count': df.count(),
        'Missing Values': df.isnull().sum(),
        'Unique Values': df.nunique()
    })
    print(info_df)
    print("-" * 80)

    # [Check 2] 연산 필수 지표들의 '값' 상태 (0, NaN, Inf)
    # 피처 엔지니어링 시 분모로 들어갈 값들이 0인지 확인하는 것이 핵심입니다.
    critical_cols = [
        'SALES_REVENUE', 'LINKED_PARTNERS', 'DEBT_RATIO', 'CASH_RATIO', 
        'EMPLOYEE_COUNT', 'OPERATING_MARGIN', 'TARGET_Y'
    ]
    
    # 존재하는 컬럼만 필터링
    existing_critical = [c for c in critical_cols if c in df.columns]
    
    print(f"2️⃣ [Critical Metrics Stats] 대상: {existing_critical}")
    stats = []
    for col in existing_critical:
        # 숫자로 변환 시도
        temp_col = pd.to_numeric(df[col], errors='coerce')
        stats.append({
            'Column': col,
            'Mean': temp_col.mean(),
            'Median': temp_col.median(),
            'Zero Count': (temp_col == 0).sum(),
            'NaN Count': temp_col.isna().sum(),
            'Max Value': temp_col.max()
        })
    print(pd.DataFrame(stats))
    print("-" * 80)

    # [Check 3] Target(Y) 라벨의 실제 분포 재확인
    if 'TARGET_Y' in df.columns:
        print("3️⃣ [Target Label Balance]")
        print(df['TARGET_Y'].value_counts(normalize=True))
        print(f"-> 실제 부실 기업 수: {df['TARGET_Y'].sum()}개")
    else:
        print("❌ 'TARGET_Y' 컬럼이 보이지 않습니다!")

except Exception as e:
    print(f"❌ 진단 중 오류 발생: {e}")

print("-" * 80)
print("✅ 진단 완료. 위 수치들을 보고 다음 단계를 결정해 주세요.")

🔍 [AUS_v2_Golden_Training_Set.csv] 정밀 진단 시작...
--------------------------------------------------------------------------------
1️⃣ [Column & Type Check]
                              Type  Non-Null Count  Missing Values  \
COMPANY_ID                   int64            1514               0   
receivable_Total_Amt       float64            1514               0   
receivable_Partner_Count   float64            1514               0   
Receivable_Concentration   float64            1514               0   
BNPL_Req_Count             float64            1514               0   
BNPL_Avg_Amt               float64            1514               0   
BNPL_Success_Rate          float64            1514               0   
LIQUIDITY_RATIO            float64            1514               0   
DEBT_RATIO                  object            1514               0   
INTEREST_COVERAGE_RATIO    float64            1514               0   
OPERATING_MARGIN           float64            1514               0   
SALES_

In [4]:
import pandas as pd
import numpy as np

print("🧪 [Custom Engineering] 진단 결과 맞춤형 정제 및 피처 생성 시작...")

# 1. 데이터 로드
df = pd.read_csv("AUS_v2_Golden_Training_Set.csv", dtype={'COMPANY_ID_NORM': str})

# 2. [수술 1] 타입 통합 및 결측치 처리 (Median 활용)
# DEBT_RATIO를 포함한 수치형 후보군 강제 변환
cols_to_fix = ['DEBT_RATIO', 'SALES_REVENUE', 'LINKED_PARTNERS', 'CASH_RATIO', 'OPERATING_MARGIN', 'EMPLOYEE_COUNT']
for col in cols_to_fix:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    # 아웃라이어가 심하므로 평균 대신 '중앙값(Median)'으로 결측치를 채웁니다.
    df[col] = df[col].fillna(df[col].median())

# 3. [수술 2] 극단적 수치 정규화 (Log Transformation)
# 매출액 등 단위가 너무 큰 지표들은 로그를 씌워 분포를 안정화합니다.
df['FE_LOG_REVENUE'] = np.log1p(df['SALES_REVENUE'])
df['FE_LOG_EMPLOYEE'] = np.log1p(df['EMPLOYEE_COUNT'])

# 4. [수술 3] 제로 값 보정 파생 변수 생성
# 분모에 +1을 더해 Zero-division을 원천 차단합니다.

# A. 공급망 고립도: 거래처당 매출 규모 (고립된 우량주 포착)
df['FE_NET_DEPENDENCY'] = df['SALES_REVENUE'] / (df['LINKED_PARTNERS'] + 1)

# B. 유동성 압박 지수: 부채 대비 현금 부족 정도
df['FE_LIQUIDITY_STRESS'] = (df['DEBT_RATIO'] + 1) / (df['CASH_RATIO'] + 1)

# C. 수익성 효율: 인당 영업이익 (영업이익률 기반 추정)
df['FE_PROFIT_EFFICIENCY'] = (df['SALES_REVENUE'] * (df['OPERATING_MARGIN']/100)) / (df['EMPLOYEE_COUNT'] + 1)

# 5. 무한대(inf) 체크 및 최종 정제
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(0, inplace=True) # 위 연산 후 남은 미세 결측치 0 처리

print(f"✅ 정제 완료: 신규 피처 {len([c for c in df.columns if c.startswith('FE_')])}종 추가")
print(f"🎯 최종 학습 준비 데이터: {df.shape[0]}행 x {df.shape[1]}열")

# 최종 데이터 저장
df.to_csv("AUS_v2_Ready_To_Train.csv", index=False)

🧪 [Custom Engineering] 진단 결과 맞춤형 정제 및 피처 생성 시작...
✅ 정제 완료: 신규 피처 5종 추가
🎯 최종 학습 준비 데이터: 1514행 x 33열
